# 🧠 Tactile SDF Reconstruction — Google Colab (Fixed)

This notebook trains a **PointNet + SIREN** model using sparse tactile contacts.

**Data Sources:**
- **Grasp Data:** Hugging Face Hub (`jack635/grasp-dataset-curated`)
- **GT Meshes:** Objaverse API

In [ ]:
# @title 🛠️ Step 1: Setup & Dependencies
!pip install trimesh rtree plotly streamlit scikit-image scikit-learn tqdm huggingface_hub objaverse
!pip install fast-simplification  # For faster mesh decimation

In [ ]:
# @title 📂 Step 2: Download Code & Data
import os
import json
import shutil
import objaverse
from huggingface_hub import snapshot_download

ROOT = "/content/tactile_project"
os.makedirs(ROOT, exist_ok=True)
%cd {ROOT}

# 1. Clone the tactile-sdf code
if not os.path.exists('code'):
    !git clone https://github.com/635jack/tactile-sdf.git code
else:
    !cd code && git pull

# 2. Download the grasp dataset
repo_id = "jack635/grasp-dataset-curated"
if not os.path.exists('grasp-dataset'):
    print(f"📥 Downloading grasp data from {repo_id}...")
    snapshot_download(repo_id=repo_id, repo_type="dataset", local_dir="grasp-dataset")

# 3. Download GT meshes from Objaverse
mapping_path = "grasp-dataset/uid_mapping.json"
with open(mapping_path) as f:
    uid_mapping = json.load(f)

target_mesh_dir = os.path.join(ROOT, "data/objaverse")
if not os.path.exists(target_mesh_dir) or len(os.listdir(target_mesh_dir)) < 50:
    print(f"📦 Downloading {len(uid_mapping)} meshes from Objaverse...")
    objects = objaverse.load_objects(uids=list(uid_mapping.values()))
    
    os.makedirs(target_mesh_dir, exist_ok=True)
    for mesh_name, uid in uid_mapping.items():
        if uid in objects:
            shutil.copy(objects[uid], os.path.join(target_mesh_dir, f"{mesh_name}.glb"))

print(f"🎉 Setup complete. Root: {ROOT}")

In [ ]:
# @title 📐 Step 3: Run SDF Preprocessing (~10 min)
# Note: Warnings about 'divide by zero' are expected with some Objaverse meshes.
%cd {ROOT}/code
!python preprocess_sdf.py \
    --n_points 50000 \
    --glb_dir {ROOT}/data/objaverse \
    --dataset_dir {ROOT}/grasp-dataset

In [ ]:
# @title 🚀 Step 4: Start Training
%cd {ROOT}/code
!python train.py \
    --epochs 300 \
    --batch_size 16 \
    --dataset_dir {ROOT}/grasp-dataset

In [ ]:
# @title 📊 Step 5: Visualize Results
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

runs = sorted(glob.glob(f'{ROOT}/code/runs/*'))
if runs:
    latest_run = runs[-1]
    img_path = os.path.join(latest_run, 'training_curves.png')
    if os.path.exists(img_path):
        plt.figure(figsize=(15, 10))
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No training runs found.")